# Student Reflection RAG — Full RAG Pipeline

* Combines the multi-label BERT classifier, FAISS retrieval, and Gemini generation into one suggest_response() function.

### Load our Multilabel BERT Classifier

In [1]:
import torch 
import pickle 
from transformers import BertTokenizer, BertForSequenceClassification

/Users/svenwu/Hustle/MachineLearning/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# We do not have to retrain the model.
model_path = "./bert_multilabel_v1"

tokenizer = BertTokenizer.from_pretrained(model_path)

model = BertForSequenceClassification.from_pretrained(model_path)

# model.eval() switches the model from training mode to evaluation (inference) mode. 
# # Doesn't run any computation - just flips a flag that changes the behavior of certain layer.
# Dropout Layers --> During training, dropout randomly zeroes out some neurons to prevent overfitting. In eval mode, dropout is disabled—all neurons are used.
model.eval()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5904.86it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [3]:
with open("./bert_multilabel_v1/emotion_names.pkl", "rb") as f:
  emotion_names = pickle.load(f)

### Reload RAG Retrieval Setup

In [4]:
import json
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [5]:
# Load synthetic data in
with open("synthetic_student_reflections.json") as f:
  example_corpus = json.load(f)

* Embed Student Texts

In [6]:
# Load the embedder. 
embedder = SentenceTransformer("all-MiniLM-L6-v2")
embedder

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7329.84it/s]


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)

In [7]:
# Get only the student_text from the JSON.
student_texts = [ex["student_text"] for ex in example_corpus]
student_texts[:3]

['I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.',
 'My group accidentally submitted the draft with all of our comments still visible. We were embarrassed at first, but looking back at some of the comments is actually pretty funny.',
 'I spent three hours working on the assignment and then found out the instructions had changed after I started. I wish someone had told us earlier because now I have to redo most of my work.']

In [8]:
# Embed all student_text entries. 
student_text_embeddings = embedder.encode(student_texts, convert_to_numpy=True)
student_text_embeddings[:5]

array([[-0.08167081, -0.0350743 ,  0.04618319, ...,  0.1345141 ,
         0.04746348, -0.05028719],
       [-0.09006829, -0.02984722,  0.03510597, ...,  0.0344655 ,
        -0.07471598,  0.04870407],
       [-0.02693345,  0.05441548,  0.0054687 , ...,  0.09743757,
        -0.13128257, -0.04685029],
       [ 0.00585679,  0.01583065,  0.02454555, ...,  0.11912089,
        -0.10991519,  0.0521571 ],
       [-0.08291283, -0.01857998,  0.03763821, ..., -0.02041946,
        -0.02709852,  0.04316133]], shape=(5, 384), dtype=float32)

In [9]:
len(student_text_embeddings)

25

In [10]:
student_text_embeddings.shape

(25, 384)

* Build the FAISS Index.

In [11]:
# Build the FAISS index.
index = faiss.IndexFlatL2(student_text_embeddings.shape[1])

In [12]:
index.add(student_text_embeddings)

### Reload Gemini Client

In [13]:
try:
  from google import genai
except ImportError:
  import subprocess
  subprocess.run(["pip", "install", "google-genai"])
  from google import genai

In [14]:
import os
from dotenv import load_dotenv

In [15]:
load_dotenv()  # read .env file

True

In [16]:
# Create a client object once that will read the API key and handle all requests through it.
client = genai.Client(api_key = os.environ["GEMINI_API_KEY"])

### Classify The Student Reflections by Emotions

In [17]:
def predict_emotion(text, model, tokenizer, emotion_names, max_length=64, threshold=0.3):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.sigmoid(outputs.logits).squeeze()      # independent per-label probabilities
    preds = (probs > threshold).int()

    predicted_indices = torch.where(preds == 1)[0].tolist()
    predicted_emotions = [emotion_names[i] for i in predicted_indices]

    return predicted_emotions

In [19]:
# Test Classifier
predict_emotion("I absolutely love this game!", model, tokenizer, emotion_names)


['love']

### The Retriever

In [20]:
def retrieve_similar(text, k=3):
  # Converts our input text into a VECTOR, using the same "all-MiniLM-L6-v2" embedding model used to build the index. 
  # Essential, as query has to live in the SAME 384-D vector space as everything stored in the index.
    query_embedding = embedder.encode([text], convert_to_numpy=True)
    
    # The search. FAISS compares query vector against all 25 stored vectors and finds the 'k' closest ones by L2 distance.
    _, indices = index.search(query_embedding, k)
    return [example_corpus[i] for i in indices[0]]

# Helper function to display results.
def display_results(results):
    for r in results:
        print(f"\n* {r['student_text']}")

In [21]:
test_text = example_corpus[0]["student_text"]
test_text

'I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.'

In [22]:
results = retrieve_similar(test_text, k=3)
display_results(results)


* I watched how my lab partner explained the experiment to everyone when we were confused. She made something complicated seem really simple, and I wish I could explain things like that.

* I don't understand why the experiment produced such a different result from what we expected. At first I thought we made a mistake, but now I'm wondering if there's something interesting happening that we haven't considered.

* I read the directions three times and still don't understand what we're supposed to include in the final section. I know I'm missing something, but I can't figure out what it is.


### Generates the Response

* Takes in the prompt built in build_prompt and feeds it to Gemini to generate a "well-informed" response.

* The generation step of RAG — everything beforeit (classification, retrieval, prompt-building) is preparing input;
  this is the only function that calls an LLM.

In [34]:
def ask_gemini(prompt: str, temperature: float = 0.7, max_tokens: int = 1500) -> str:
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=genai.types.GenerateContentConfig(
            temperature=temperature,
            max_output_tokens=max_tokens,
        ),
    )
    return response.text

### Build The Prompt

* Combines 3 inputs into one finished prompt string:
  - The student's raw text
  - The classified emotion(s) from predict_emotion
  - The retrieved similar examples from retrieve_similar

* Does NOT call Gemini itself — this function only builds the prompt text. 
  - ask_gemini() is what sends it to the LLM afterward.

* The augmentation step of RAG — retrieved context gets inserted into the prompt here, before generation happens.

In [35]:
def build_prompt(text, emotions, similar_examples):
    examples_block = "\n\n".join(
        f"Student wrote: \"{ex['student_text']}\"\n"
        f"Detected emotions: {', '.join(ex['emotions'])}\n"
        f"Teacher responded: \"{ex['teacher_response']}\""
        for ex in similar_examples
    )

    emotions_str = ", ".join(emotions) if emotions else "unclear / mixed"

    return f"""You are helping a teacher respond to a student's written reflection.

Student's reflection: "{text}"
Detected emotions: {emotions_str}

Here are similar past cases and how a teacher responded to each:

{examples_block}

Using the detected emotions and the tone/approach shown in the similar cases above, suggest a thoughtful, supportive response a teacher could give this student. Keep it warm, specific to what the student wrote, and no longer than 3-4 sentences."""

### Suggest Response 

* Get an augmented response from our RAG pipeline.

* The orchestrator function — calls all 4 pieces in sequence to
  produce one final teacher-facing suggestion:

  1. predict_emotion()   → classify the student's text (BERT)
  2. retrieve_similar()  → find similar past examples (FAISS)
  3. build_prompt()      → combine both into one prompt string
  4. ask_gemini()        → generate the final response (LLM)


* This is the full Retrieval-Augmented Generation pipeline end to
  end: Retrieval (step 2) + Augmentation (step 3) + Generation
  (step 4), with classification (step 1) as an extra input feeding
  the augmentation step alongside retrieval.

* Takes a single argument (raw student text) and returns a single
  string (the suggested response) — everything else is handled
  internally, nothing else needs to be passed in by the caller.

In [36]:
def suggest_response(text):
    emotions = predict_emotion(text, model, tokenizer, emotion_names)
    similar_examples = retrieve_similar(text)
    prompt = build_prompt(text, emotions, similar_examples)
    return ask_gemini(prompt)

In [38]:
test_text = "I got a B+ on the essay but I know I could have done better if I'd started earlier"
result = suggest_response(test_text)
print(result)

Here is a thoughtful and supportive response you could give the student:

"It sounds like you're feeling a mix of pride in earning a solid B+ and frustration knowing you could have achieved even more with a head start. Recognizing how your timeline impacts the quality of your work shows great self-awareness as a learner. Next time, giving yourself a little extra time early on will let you see just how far your potential can really take you."


In [39]:
emotions = predict_emotion(test_text, model, tokenizer, emotion_names)
print(emotions)

[]


In [40]:
inputs = tokenizer(test_text, return_tensors="pt", truncation=True, max_length=64)
with torch.no_grad():
    outputs = model(**inputs)
probs = torch.sigmoid(outputs.logits).squeeze()

# Show top 5 emotions by probability, even if none cross 0.3
top5 = torch.topk(probs, 5)
for i, p in zip(top5.indices.tolist(), top5.values.tolist()):
    print(f"{emotion_names[i]}: {p:.3f}")

realization: 0.276
disappointment: 0.263
neutral: 0.177
approval: 0.128
optimism: 0.091


### Response Evaluation:

* Example of the global-threshold limitation: 
  - "B+ but could have done better" scored realization=0.276, disappointment=0.263

  - Both directionally correct but just under the 0.3 cutoff, so predict_emotion() returns []. 
  
  - Illustrates why per-class thresholds (flagged as future work by both the reference paper and our own
  threshold sweep) would likely help — a single global cutoff can't be optimal for every class simultaneously.

In [ ]:
test_texts = [
    # 1. Clear single emotion — sanity check, should work cleanly
    "I'm so excited for the science fair next week, I've been looking forward to it all month!",

    # 2. Genuine multi-label case — tests whether multiple emotions surface together
    "I'm proud that I finally finished the project, but I'm also nervous about presenting it in front of everyone tomorrow.",

    # 3. Subtle/understated — similar territory to your B+ example, likely to test threshold sensitivity again
    "I turned in my essay on time for once. I guess that's something.",

    # 4. Rare/harder emotion — tests a class your model struggled with in eval (embarrassment had decent support but was still mid-tier F1)
    "Everyone laughed when I mispronounced the word out loud during my presentation, and I just wanted to disappear.",

    # 5. Negative/frustration-adjacent, tests annoyance/anger vs disappointment distinction
    "I asked my group members three times to send their part and they still haven't, now we're going to submit late because of them.",
]

for i, text in enumerate(test_texts, 1):
    print(f"\n{'='*60}")
    print(f"Test {i}: {text}")
    print(f"{'='*60}")

    emotions = predict_emotion(text, model, tokenizer, emotion_names)
    print(f"Detected emotions: {emotions if emotions else '(none — below threshold)'}")

    response = suggest_response(text)
    print(f"\nSuggested response:\n{response}")


Test 1: I'm so excited for the science fair next week, I've been looking forward to it all month!
Detected emotions: ['excitement']


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 7.5905785s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'}, 'q